# GP1 ASR - Efficient Conformer on Google Colab

End-to-end pipeline: clone repo -> install extras -> download Kaggle data -> dev split -> HP search (random, N=20) -> predict -> `submission.csv`.

**Recommended runtime:** Runtime > Change runtime type > GPU (T4 Free, or A100/L4 on Pro).

**Kaggle credentials:** either
- upload `kaggle.json` via the file picker below (easiest), OR
- store `KAGGLE_USERNAME` + `KAGGLE_KEY` in Colab Secrets (key icon on the left sidebar) and switch to the commented-out cell.

## 1. Clone repository

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git"
REPO_DIR = Path("/content/ITMO_Speech_Recognition_Course")
GP1_ROOT = REPO_DIR / "group-projects" / "gp1"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

print("Repo:", REPO_DIR)
print("GP1 root:", GP1_ROOT)
assert (GP1_ROOT / "scripts" / "train.py").exists()

## 2. Install extras

Colab ships torch + torchaudio. We add only the rest.

In [ ]:
!pip install -q num2words pyctcdecode jiwer "pyyaml>=6.0" tqdm librosa soundfile

## 3. Mount Google Drive

Prepare once on your own machine: upload a folder named `gp1_data/` to your Google Drive root with the same layout as local:

```
gp1_data/
  train/train.csv
  train/train/*.wav   (or *.mp3)
  dev/dev.csv
  dev/dev/*.wav
  test/test.csv
  test/test/*.mp3
```

The next cell mounts Drive and points `DATA_ROOT` at that folder. No Kaggle API / auth tokens needed.

In [ ]:
from google.colab import drive  # type: ignore
from pathlib import Path

drive.mount("/content/drive")

# Edit if your folder name differs.
DATA_ROOT = Path("/content/drive/MyDrive/gp1_data")

assert (DATA_ROOT / "train" / "train.csv").exists(), (
    f"Missing: {DATA_ROOT}/train/train.csv — upload data to your Drive"
)
assert (DATA_ROOT / "dev" / "dev.csv").exists(), (
    f"Missing: {DATA_ROOT}/dev/dev.csv"
)
print("DATA_ROOT:", DATA_ROOT)
for p in sorted(DATA_ROOT.iterdir()):
    print("  ", p.name + ("/" if p.is_dir() else ""))


In [ ]:
TRAIN_CSV      = DATA_ROOT / "train" / "train.csv"
TRAIN_WAV_ROOT = DATA_ROOT / "train"
DEV_CSV        = DATA_ROOT / "dev" / "dev.csv"
DEV_WAV_ROOT   = DATA_ROOT / "dev"
TEST_CSV       = DATA_ROOT / "test" / "test.csv"
TEST_WAV_ROOT  = DATA_ROOT / "test"

for p in [TRAIN_CSV, TRAIN_WAV_ROOT, DEV_CSV, DEV_WAV_ROOT]:
    assert p.exists(), f"Missing: {p}"

import csv as _csv
with open(TRAIN_CSV, encoding="utf-8") as fh:
    n_train = sum(1 for _ in _csv.DictReader(fh))
with open(DEV_CSV, encoding="utf-8") as fh:
    n_dev = sum(1 for _ in _csv.DictReader(fh))
print(f"train: {n_train} samples -> {TRAIN_CSV}")
print(f"dev:   {n_dev} samples -> {DEV_CSV}")


## 6. (Optional) Mount Google Drive for checkpoint persistence

Colab sessions are ephemeral. To keep checkpoints across sessions, mount Drive and point `OUTPUT_BASE` into it.

In [ ]:
MOUNT_DRIVE = False  # flip to True to persist runs in Drive

if MOUNT_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/gp1_runs/efficient_conformer")
else:
    OUTPUT_BASE = Path("/content/runs/efficient_conformer")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print("OUTPUT_BASE:", OUTPUT_BASE)

## HP Search (Random, N=20)

Random search over a 5-dimensional hyperparameter space following Bergstra & Bengio (2012) JMLR — 20 independent trials, each a full training run. The formula N >= log(1-p)/log(1-alpha) guarantees p=0.95 probability of finding a configuration in the top alpha fraction of the search space with N trials.

In [ ]:
import math, json, yaml, sys

BASELINE = "efficient_conformer"
CONFIG_NAME = "efficient_conformer.yaml"
N_TRIALS = 20
RANDOM_SEED = 42

DEVICE = "cuda"
NUM_WORKERS = 4


def sample_params(rng: random.Random) -> dict:
    """5-dim search space per Bergstra & Bengio (2012)."""
    return {
        "lr": 10 ** rng.uniform(-4, math.log10(5e-3)),
        "dropout": rng.uniform(0.05, 0.25),
        "warmup_steps": rng.randint(500, 8000),
        "grad_clip_norm": rng.choice([0.5, 1.0, 2.0, 5.0]),
        "specaug_time_mask_param": rng.choice([15, 25, 35, 50]),
    }


def patch_config(base_path: Path, params: dict, out_path: Path) -> None:
    """Load base config, merge defaults, apply HP params, write patched YAML."""
    with open(base_path) as f:
        cfg = yaml.safe_load(f)
    if "defaults" in cfg:
        for name in cfg.pop("defaults"):
            with open(base_path.parent / f"{name}.yaml") as f:
                base = yaml.safe_load(f)
            merged = {**base}
            for k, v in cfg.items():
                merged[k] = (
                    {**merged[k], **v}
                    if isinstance(v, dict) and isinstance(merged.get(k), dict)
                    else v
                )
            cfg = merged
    cfg.setdefault("train", {})
    cfg["train"]["lr"] = params["lr"]
    cfg["train"].setdefault("optimizer", {})
    cfg["train"]["optimizer"]["lr"] = params["lr"]
    cfg["train"].setdefault("scheduler", {})
    cfg["train"]["scheduler"]["warmup_steps"] = params["warmup_steps"]
    cfg["train"]["grad_clip_norm"] = params["grad_clip_norm"]
    cfg.setdefault("model", {})
    cfg["model"]["dropout"] = params["dropout"]
    cfg.setdefault("aug", {})
    cfg["aug"]["specaug_time_mask_param"] = params["specaug_time_mask_param"]
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f)

In [ ]:
rng = random.Random(RANDOM_SEED)
trials_dir = OUTPUT_BASE / "hp_search"
trials_dir.mkdir(parents=True, exist_ok=True)
trials_log = trials_dir / "trials.jsonl"
best = {"trial_id": None, "cer": float("inf"), "ckpt": None, "params": None}

for trial_id in range(N_TRIALS):
    params = sample_params(rng)
    trial_dir = trials_dir / f"trial_{trial_id:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    patched = trial_dir / "config.yaml"
    patch_config(GP1_ROOT / "configs" / CONFIG_NAME, params, patched)
    cmd = [
        sys.executable, "scripts/train.py",
        "--config", str(patched),
        "--train-csv", str(TRAIN_CSV), "--train-root", str(TRAIN_WAV_ROOT),
        "--dev-csv", str(DEV_CSV), "--dev-root", str(TRAIN_WAV_ROOT),
        "--output-dir", str(trial_dir),
        "--num-workers", str(NUM_WORKERS),
        "--device", DEVICE,
    ]
    try:
        subprocess.run(cmd, cwd=str(GP1_ROOT), check=True)
        result = json.loads((trial_dir / "result.json").read_text())
        cer = float(result["best_val_cer"])
    except Exception as exc:
        cer = float("inf")
        result = {"error": str(exc)}
    with open(trials_log, "a") as f:
        f.write(json.dumps({"trial_id": trial_id, "params": params, "best_val_cer": cer}) + "\n")
    if cer < best["cer"]:
        best = {"trial_id": trial_id, "cer": cer, "ckpt": result.get("best_ckpt_path"), "params": params}
    print(f"trial {trial_id:03d}: CER={cer:.4f}  lr={params['lr']:.2e}")

print(f"\nBEST trial_id={best['trial_id']} CER={best['cer']:.4f}  ckpt={best['ckpt']}")
(trials_dir / "best_trial.json").write_text(json.dumps(best, indent=2, default=str))

## 7. Predict -> submission.csv

In [ ]:
SUBMISSION_CSV = Path("/content/submission.csv")
cmd = [
    sys.executable, "scripts/predict.py",
    "--checkpoint", best["ckpt"],
    "--config", str(trials_dir / f"trial_{best['trial_id']:03d}" / "config.yaml"),
    "--test-csv", str(TEST_CSV), "--test-root", str(TEST_WAV_ROOT),
    "--output", str(SUBMISSION_CSV),
    "--device", DEVICE,
]
print("$", " ".join(cmd))
subprocess.run(cmd, cwd=str(GP1_ROOT), check=True)
print("Wrote:", SUBMISSION_CSV)

## 8. Download submission.csv

In [ ]:
import pandas as pd
from google.colab import files  # type: ignore

df = pd.read_csv(SUBMISSION_CSV)
print(df.shape)
print(df.head(10))
files.download(str(SUBMISSION_CSV))